In [1]:
from pathlib import Path
import importlib.metadata
import json
import sys

from google.colab import drive

drive.mount("/content/drive")

drive_root = Path("/content/drive/MyDrive/FacialVisualProfile")
assert drive_root.is_dir(), f"Missing project directory: {drive_root}"

print("Python:", sys.version.split()[0])

for package in (
    "torch", "torchvision", "facenet-pytorch", "Pillow", "numpy"
):
    try:
        print(f"{package}: {importlib.metadata.version(package)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

for run_name in ("r18_seed0", "r50_seed0", "m3_seed0"):
    run_path = drive_root / "runs" / run_name
    best_path = run_path / "best.json"
    assert best_path.is_file(), f"Missing: {best_path}"

    best = json.loads(best_path.read_text(encoding="utf-8"))
    checkpoint_path = run_path / best["checkpoint"]
    assert checkpoint_path.is_file(), f"Missing: {checkpoint_path}"

    print(
        f"{run_name}: epoch={best['epoch']}, "
        f"checkpoint={best['checkpoint']}; FOUND"
    )

detector_candidates = [
    drive_root / "detectors/mtcnn.py",
    Path("/content/facial-visual-profile/detectors/mtcnn.py"),
]
for path in detector_candidates:
    print("Detector source:", path, "| exists:", path.is_file())

print("Day 16 environment and checkpoint inspection: PASS")

Mounted at /content/drive
Python: 3.13.16
torch: 2.11.0+cpu
torchvision: 0.26.0+cpu
facenet-pytorch: NOT INSTALLED
Pillow: 11.3.0
numpy: 2.1.3
CUDA available: False
r18_seed0: epoch=3, checkpoint=epoch_03.pt; FOUND
r50_seed0: epoch=3, checkpoint=epoch_03.pt; FOUND
m3_seed0: epoch=2, checkpoint=epoch_02.pt; FOUND
Detector source: /content/drive/MyDrive/FacialVisualProfile/detectors/mtcnn.py | exists: False
Detector source: /content/facial-visual-profile/detectors/mtcnn.py | exists: False
Day 16 environment and checkpoint inspection: PASS


In [2]:
from google.colab import files

uploaded = files.upload()
assert len(uploaded) == 1
assert "day16_validation_inputs.zip" in uploaded

print("Day 16 input ZIP uploaded.")

Saving day16_validation_inputs.zip to day16_validation_inputs.zip
Day 16 input ZIP uploaded.


In [3]:
%pip install --no-deps facenet-pytorch==2.6.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 66.1 MB/s eta 0:00:00


In [4]:
import importlib.metadata
from facenet_pytorch import MTCNN

print("facenet-pytorch:", importlib.metadata.version("facenet-pytorch"))

detector_smoke = MTCNN(device="cpu").eval()
print("MTCNN initialization: PASS")
del detector_smoke

facenet-pytorch: 2.6.0
MTCNN initialization: PASS


In [5]:
from pathlib import Path
from collections import Counter
import hashlib
import importlib.metadata
import importlib.util
import json
import shutil
import time
import zipfile

import numpy as np
from PIL import Image
import torch

drive_root = Path("/content/drive/MyDrive/FacialVisualProfile")
zip_path = Path("/content/day16_validation_inputs.zip")
assert zip_path.is_file(), "Upload day16_validation_inputs.zip first."

work_dir = Path("/content/day16_validation_inputs")
report_dir = drive_root / "runs/corruption_preparation/day16"
work_dir.mkdir(parents=True, exist_ok=True)
report_dir.mkdir(parents=True, exist_ok=True)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def rgb_sha256(image):
    header = f"{image.mode}:{image.width}:{image.height}:".encode("ascii")
    return hashlib.sha256(header + image.tobytes()).hexdigest()

# Extract only the expected package members.
with zipfile.ZipFile(zip_path) as archive:
    manifest = json.loads(archive.read("manifest.json"))
    assert manifest["split"] == "validation"
    assert manifest["test_evaluated"] is False
    assert len(manifest["records"]) == 84
    assert len(manifest["sample_filenames"]) == 4

    expected_members = {
        "manifest.json",
        "evidence/day14_report.json",
        "evidence/day15_report.json",
        *[
            row["relative_png"]
            for row in manifest["records"]
        ],
        *[
            "source/" + filename
            for filename in manifest["source_sha256"]
        ],
    }
    assert len(archive.namelist()) == len(expected_members)
    assert set(archive.namelist()) == expected_members

    for filename in expected_members:
        relative = Path(filename)
        assert not relative.is_absolute() and ".." not in relative.parts
        destination = work_dir / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(archive.read(filename))

assert sha256(work_dir / "evidence/day14_report.json") == (
    manifest["day14_report_sha256"]
)
assert sha256(work_dir / "evidence/day15_report.json") == (
    manifest["day15_report_sha256"]
)

for filename, expected in manifest["source_sha256"].items():
    assert sha256(work_dir / "source" / filename) == expected

# Verify every image before starting detection.
keys = set()
for row in manifest["records"]:
    key = (row["sample_id"], row["condition"], row["severity"])
    assert key not in keys
    keys.add(key)

    path = work_dir / row["relative_png"]
    assert sha256(path) == row["png_file_sha256"]
    with Image.open(path) as image:
        image = image.convert("RGB")
        assert image.size == (224, 224)
        assert rgb_sha256(image) == row["output_rgb_sha256"]

expected_keys = {
    (name, condition, severity)
    for name in manifest["sample_filenames"]
    for condition, severity in (
        [("clean", 0)]
        + [
            (kind, level)
            for kind in ("blur", "brightness", "rotation", "occlusion", "jpeg")
            for level in range(1, 5)
        ]
    )
}
assert keys == expected_keys

# Preserve the exact input package on Drive.
saved_zip = report_dir / "day16_validation_inputs.zip"
if saved_zip.exists():
    assert sha256(saved_zip) == sha256(zip_path), (
        "A different Day 16 input package already exists."
    )
else:
    shutil.copy2(zip_path, saved_zip)

print("84 input images and source hashes: PASS")

# Load the uploaded wrapper directly, without changing its source.
detector_source = work_dir / "source/detectors/mtcnn.py"
spec = importlib.util.spec_from_file_location(
    "day16_saved_mtcnn", detector_source
)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

expected_order = (
    "lefteye", "righteye", "nose", "leftmouth", "rightmouth"
)
assert tuple(module.MTCNNDetector.POINT_ORDER) == expected_order

torch.set_num_threads(2)
detector = module.MTCNNDetector(device="cpu")

# Record the actual loaded detector weights.
weight_hash = hashlib.sha256()
for name, tensor in sorted(detector.model.state_dict().items()):
    value = tensor.detach().cpu().contiguous().numpy()
    weight_hash.update(name.encode("utf-8"))
    weight_hash.update(str(value.dtype).encode("ascii"))
    weight_hash.update(str(value.shape).encode("ascii"))
    weight_hash.update(value.tobytes())

def calculate_nme(predicted, gt):
    predicted = np.asarray(predicted, dtype=np.float64)
    gt = np.asarray(gt, dtype=np.float64)
    assert predicted.shape == gt.shape == (5, 2)
    assert np.isfinite(predicted).all() and np.isfinite(gt).all()
    eye_distance = float(np.linalg.norm(gt[0] - gt[1]))
    assert eye_distance > 0
    errors = np.linalg.norm(predicted - gt, axis=1)
    return float(errors.mean() / eye_distance), eye_distance

# Analytic check: translating all points by (3,4) gives 5px error.
example_gt = np.array([
    [0, 0], [10, 0], [5, 5], [2, 8], [8, 8]
], dtype=np.float64)
assert calculate_nme(example_gt, example_gt)[0] == 0.0
assert np.isclose(
    calculate_nme(example_gt + [3, 4], example_gt)[0], 0.5
)

results = []
started = time.perf_counter()

for index, row in enumerate(manifest["records"], start=1):
    gt = np.asarray(row["gt_pixels"], dtype=np.float64)
    assert gt.shape == (5, 2) and np.isfinite(gt).all()
    eye_distance = float(np.linalg.norm(gt[0] - gt[1]))
    assert eye_distance > 0

    path = work_dir / row["relative_png"]
    with Image.open(path) as image:
        image = image.convert("RGB")
        try:
            detection = detector.detect(image)
        except Exception as error:
            detection = {
                "status": "runtime_error",
                "landmarks": None,
                "error_type": type(error).__name__,
                "error": str(error),
            }

    status = detection["status"]
    assert status in ("ok", "no_face", "invalid_output", "runtime_error")

    valid = status == "ok"
    nme = None
    normalized_points = [0.0] * 10

    if valid:
        predicted = np.asarray(detection["landmarks"], dtype=np.float64)
        nme, eye_distance = calculate_nme(predicted, gt)
        normalized_points = (
            2 * predicted / 223 - 1
        ).reshape(-1).tolist()

    results.append({
        "sample_id": row["sample_id"],
        "condition": row["condition"],
        "severity": row["severity"],
        "relative_png": row["relative_png"],
        "input_rgb_sha256": row["output_rgb_sha256"],
        "corruption_parameters": row["parameters"],
        "gt_pixels": gt.tolist(),
        "gt_eye_distance_px": eye_distance,
        "geometry_valid": valid,
        "predicted_normalized_points": normalized_points,
        "nme": nme,
        "detection": detection,
    })

    if not valid:
        print(
            f"Detection status: {row['sample_id']} "
            f"{row['condition']} s{row['severity']} -> {status}"
        )
    if index % 4 == 0:
        print(f"Processed {index}/84", flush=True)

summaries = []
conditions = [
    ("clean", 0),
    *[
        (kind, level)
        for kind in ("blur", "brightness", "rotation", "occlusion", "jpeg")
        for level in range(1, 5)
    ],
]

print("\n=== Detector quality summary ===")
for condition, severity in conditions:
    subset = [
        row for row in results
        if row["condition"] == condition and row["severity"] == severity
    ]
    assert len(subset) == 4
    counts = Counter(row["detection"]["status"] for row in subset)
    valid_nme = [row["nme"] for row in subset if row["geometry_valid"]]
    failures = counts["no_face"] + counts["invalid_output"]

    summary = {
        "condition": condition,
        "severity": severity,
        "total": len(subset),
        "valid": counts["ok"],
        "no_face": counts["no_face"],
        "invalid_output": counts["invalid_output"],
        "runtime_errors": counts["runtime_error"],
        "coverage": counts["ok"] / len(subset),
        "failure_rate": failures / len(subset),
        "nme_count": len(valid_nme),
        "mean_nme": float(np.mean(valid_nme)) if valid_nme else None,
        "median_nme": float(np.median(valid_nme)) if valid_nme else None,
    }
    summaries.append(summary)

    nme_text = (
        f"{100 * summary['mean_nme']:.2f}%"
        if valid_nme else "N/A"
    )
    print(
        f"{condition:10s} s{severity}: "
        f"valid={summary['valid']}/4, "
        f"failures={failures}, errors={summary['runtime_errors']}, "
        f"mean NME={nme_text}"
    )

runtime_errors = sum(
    row["detection"]["status"] == "runtime_error" for row in results
)

report = {
    "split": "validation",
    "sample_filenames": manifest["sample_filenames"],
    "unique_inputs": len(results),
    "input_package_sha256": sha256(zip_path),
    "source_sha256": manifest["source_sha256"],
    "detector_config": detector.CONFIG,
    "point_order": list(detector.POINT_ORDER),
    "detector_weights_sha256": weight_hash.hexdigest(),
    "environment": {
        package: importlib.metadata.version(package)
        for package in (
            "torch", "torchvision", "facenet-pytorch", "Pillow", "numpy"
        )
    },
    "device": "cpu",
    "torch_threads": torch.get_num_threads(),
    "elapsed_seconds": time.perf_counter() - started,
    "nme_definition": "mean five-point Euclidean error / GT eye distance",
    "nme_unit": "ratio; printed as percent",
    "failure_policy": "retain sample; zero geometry input with validity mask",
    "runtime_errors": runtime_errors,
    "summaries": summaries,
    "records": results,
    "model_inference_run": False,
    "test_evaluated": False,
    "benchmark_frozen": False,
    "scope": (
        "Four-sample detector pipeline check per condition, not a benchmark "
        "performance estimate. Runtime errors are reported separately from "
        "detector failures. Actual M3 mask inference remains pending."
    ),
    "passed": runtime_errors == 0,
}

report_path = report_dir / "detector_quality_report.json"
temporary = report_path.with_suffix(".json.tmp")
temporary.write_text(
    json.dumps(report, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)
temporary.replace(report_path)

print("\nSaved:", report_path)
assert runtime_errors == 0, (
    "Runtime errors occurred; inspect their messages before proceeding."
)
print("Day 16 detector execution and statistics: PASS")

84 input images and source hashes: PASS
Processed 4/84
Processed 8/84
Processed 12/84
Processed 16/84
Processed 20/84
Processed 24/84
Processed 28/84
Processed 32/84
Processed 36/84
Processed 40/84
Processed 44/84
Processed 48/84
Processed 52/84
Processed 56/84
Processed 60/84
Processed 64/84
Processed 68/84
Processed 72/84
Processed 76/84
Detection status: 182637.jpg occlusion s3 -> no_face
Detection status: 182637.jpg occlusion s4 -> no_face
Processed 80/84
Processed 84/84

=== Detector quality summary ===
clean      s0: valid=4/4, failures=0, errors=0, mean NME=4.42%
blur       s1: valid=4/4, failures=0, errors=0, mean NME=4.93%
blur       s2: valid=4/4, failures=0, errors=0, mean NME=5.68%
blur       s3: valid=4/4, failures=0, errors=0, mean NME=6.24%
blur       s4: valid=4/4, failures=0, errors=0, mean NME=5.88%
brightness s1: valid=4/4, failures=0, errors=0, mean NME=4.31%
brightness s2: valid=4/4, failures=0, errors=0, mean NME=4.69%
brightness s3: valid=4/4, failures=0, errors=

In [6]:
from pathlib import Path
import torch

root = Path("/content/drive/MyDrive/FacialVisualProfile")

# 查找训练时保存的 M3 模型源码及最佳 checkpoint。
sources = sorted(root.rglob("resnet18_geometry.py"))
checkpoints = sorted(
    p for p in root.rglob("epoch_02.pt")
    if "m3_seed0" in str(p)
)

print("M3 source candidates:")
for p in sources:
    print(p)

print("\nM3 checkpoint candidates:")
for p in checkpoints:
    print(p)

assert checkpoints, "没有找到 m3_seed0 的 epoch_02.pt"

for path in checkpoints:
    # 仅加载你自己训练并保存的可信 checkpoint。
    checkpoint = torch.load(
        path, map_location="cpu", weights_only=False
    )
    print("\nCheckpoint:", path)
    print("Top-level keys:", list(checkpoint))

    for key in ("epoch", "best_epoch", "config", "model_config"):
        if key in checkpoint:
            print(f"{key}:", checkpoint[key])

    state = checkpoint.get("model_state_dict")
    assert state is not None, "缺少预期的 model_state_dict"
    print("State entries:", len(state))
    print("First 12 parameter names:", list(state)[:12])

for path in sources:
    print("\nModel source:", path)
    print(path.read_text(encoding="utf-8"))

M3 source candidates:
/content/drive/MyDrive/FacialVisualProfile/runs/m3_preparation/day10/source/models/resnet18_geometry.py
/content/drive/MyDrive/FacialVisualProfile/runs/m3_preparation/day11/source/models/resnet18_geometry.py
/content/drive/MyDrive/FacialVisualProfile/runs/m3_seed0/source/models/resnet18_geometry.py

M3 checkpoint candidates:
/content/drive/MyDrive/FacialVisualProfile/runs/m3_seed0/epoch_02.pt

Checkpoint: /content/drive/MyDrive/FacialVisualProfile/runs/m3_seed0/epoch_02.pt
Top-level keys: ['epoch', 'config', 'model_state_dict', 'optimizer_state_dict', 'history', 'best']
epoch: 2
config: {'model': 'ResNet18GeometryAttributes', 'seed': 0, 'epochs': 10, 'batch_size': 64, 'num_workers': 2, 'optimizer': 'Adam', 'learning_rate': 0.0001, 'weight_decay': 0.0, 'precision': 'FP32', 'initial_weights': 'ResNet18_Weights.IMAGENET1K_V1', 'augmentation': 'None; use existing shared preprocessing', 'f1_threshold': 0.5, 'selection_metric': 'validation_mAP', 'selection_tie_rule': 'e

In [7]:
from pathlib import Path
import hashlib
import importlib.util
import importlib.metadata
import json
import sys
import types
import zipfile

import torch
from PIL import Image

root = Path("/content/drive/MyDrive/FacialVisualProfile")
inputs = Path("/content/day16_validation_inputs")
output_dir = root / "runs/corruption_preparation/day16"

detector_report_path = output_dir / "detector_quality_report.json"
checkpoint_path = root / "runs/m3_seed0/epoch_02.pt"
source_path = (
    root / "runs/m3_seed0/source/models/resnet18_geometry.py"
)


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


manifest_path = inputs / "manifest.json"
manifest = json.loads(manifest_path.read_text())
quality = json.loads(detector_report_path.read_text())

assert quality["passed"] is True
assert quality["test_evaluated"] is False
assert quality["runtime_errors"] == 0
assert len(quality["records"]) == 84

# 将本次推理绑定到已归档的 Day 16 输入包。
package_path = output_dir / "day16_validation_inputs.zip"
assert file_sha256(package_path) == quality["input_package_sha256"]
with zipfile.ZipFile(package_path) as archive:
    assert archive.read("manifest.json") == manifest_path.read_bytes()

for filename, expected in manifest["source_sha256"].items():
    assert file_sha256(inputs / "source" / filename) == expected, filename

checkpoint_sha256 = file_sha256(checkpoint_path)

# 只加载自己训练并保存的可信 checkpoint。
checkpoint = torch.load(
    checkpoint_path, map_location="cpu", weights_only=False
)
config = checkpoint["config"]
assert checkpoint["epoch"] == 2
assert config["model"] == "ResNet18GeometryAttributes"
assert len(config["attribute_names"]) == 24

expected_source = config["source_sha256"]["models/resnet18_geometry.py"]
assert file_sha256(source_path) == expected_source

# 检查输入包的共享预处理仍与训练时完全一致。
preprocessing_path = inputs / "source/datasets/preprocessing.py"
assert file_sha256(preprocessing_path) == (
    config["source_sha256"]["datasets/preprocessing.py"]
)

model_module = load_module("_day16_m3_trained", source_path)
torch.manual_seed(0)
torch.set_num_threads(2)

model = model_module.ResNet18GeometryAttributes(
    num_attributes=24, pretrained=False
).cpu().eval()
model.load_state_dict(checkpoint["model_state_dict"], strict=True)

# 独立模块名，避免 Colab 中已有 datasets 包造成导入冲突。
package_name = "_day16_verified_datasets"
package = types.ModuleType(package_name)
package.__path__ = [str(inputs / "source/datasets")]
sys.modules[package_name] = package

benchmark = load_module(
    package_name + ".benchmark_preprocessing",
    inputs / "source/datasets/benchmark_preprocessing.py",
)

print("Training source hash and strict checkpoint loading: PASS")


def condition_key(row):
    return row["sample_id"], row["condition"], row["severity"]


manifest_rows = {
    condition_key(row): row for row in manifest["records"]
}
rows = quality["records"]
assert len(manifest_rows) == 84
assert len({condition_key(row) for row in rows}) == 84
assert {condition_key(row) for row in rows} == set(manifest_rows)

images, coordinates, validity = [], [], []

for row in rows:
    original = manifest_rows[condition_key(row)]
    assert row["relative_png"] == original["relative_png"]

    image_path = inputs / row["relative_png"]
    assert file_sha256(image_path) == original["png_file_sha256"]

    with Image.open(image_path) as image:
        image = image.convert("RGB")
        assert image.size == (224, 224)
        rgb_hash = hashlib.sha256(
            b"RGB:224:224:" + image.tobytes()
        ).hexdigest()
        assert rgb_hash == row["input_rgb_sha256"]
        images.append(benchmark.normalize_canvas(image))

    valid = row["geometry_valid"]
    assert isinstance(valid, bool)
    assert valid == (row["detection"]["status"] == "ok")

    points = torch.tensor(
        row["predicted_normalized_points"], dtype=torch.float32
    )
    assert points.shape == (10,)
    assert torch.isfinite(points).all()

    if valid:
        detected = torch.tensor(
            row["detection"]["landmarks"], dtype=torch.float32
        )
        reconstructed = benchmark.normalize_points(detected)
        torch.testing.assert_close(
            points, reconstructed, rtol=1e-6, atol=1e-6
        )
    else:
        assert torch.equal(points, torch.zeros(10))

    coordinates.append(points)
    validity.append(valid)

images = torch.stack(images)
coordinates = torch.stack(coordinates)
validity = torch.tensor(validity, dtype=torch.bool)

assert validity.any() and (~validity).any()

# 捕获真正进入分类器的融合特征，检查最后 32 维几何特征。
captured = {}


def capture_classifier_input(module, args):
    captured["fusion"] = args[0].detach().clone()


hook = model.classifier.register_forward_pre_hook(
    capture_classifier_input
)

predictions = []
inference_rows = []

try:
    with torch.inference_mode():
        for start in range(0, len(rows), 4):
            stop = min(start + 4, len(rows))
            batch_images = images[start:stop]
            batch_points = coordinates[start:stop]
            batch_valid = validity[start:stop]

            logits = model(
                batch_images, batch_points, batch_valid
            )
            assert logits.shape == (stop - start, 24)
            assert torch.isfinite(logits).all()

            geometry = captured["fusion"][:, -32:]
            if (~batch_valid).any():
                assert torch.count_nonzero(
                    geometry[~batch_valid]
                ).item() == 0

            if batch_valid.any():
                expected = model.geometry_encoder(
                    batch_points[batch_valid]
                )
                torch.testing.assert_close(
                    geometry[batch_valid], expected,
                    rtol=1e-6, atol=1e-6,
                )

            probabilities = torch.sigmoid(logits)
            assert torch.isfinite(probabilities).all()
            predictions.append(logits.clone())

            for offset, row in enumerate(rows[start:stop]):
                inference_rows.append({
                    "sample_id": row["sample_id"],
                    "condition": row["condition"],
                    "severity": row["severity"],
                    "geometry_valid": row["geometry_valid"],
                    "logits": logits[offset].tolist(),
                    "probabilities": probabilities[offset].tolist(),
                })

            if stop % 16 == 0 or stop == len(rows):
                print(f"M3 inference: {stop}/{len(rows)}")

        # 同一个混合 batch：修改失败样本坐标，输出应保持一致。
        valid_indices = validity.nonzero().flatten()[:2]
        invalid_indices = (~validity).nonzero().flatten()
        indices = torch.cat([valid_indices, invalid_indices])

        x = images[indices]
        g = coordinates[indices]
        mask = validity[indices]
        reference = model(x, g, mask)

        mask_errors = {}
        for label, value in (("large_finite", 123.0), ("nan", float("nan"))):
            changed = g.clone()
            changed[~mask] = value
            actual = model(x, changed, mask)

            assert torch.isfinite(actual).all()
            torch.testing.assert_close(
                actual, reference, rtol=1e-6, atol=1e-6
            )
            assert torch.count_nonzero(
                captured["fusion"][~mask, -32:]
            ).item() == 0

            mask_errors[label] = (
                actual - reference
            ).abs().max().item()
finally:
    hook.remove()

all_logits = torch.cat(predictions)
assert all_logits.shape == (84, 24)

report = {
    "split": "validation",
    "sample_filenames": quality["sample_filenames"],
    "input_count": 84,
    "valid_geometry_count": int(validity.sum().item()),
    "masked_geometry_count": int((~validity).sum().item()),
    "checkpoint_path": str(checkpoint_path),
    "checkpoint_epoch": checkpoint["epoch"],
    "checkpoint_sha256": checkpoint_sha256,
    "model_source_sha256": file_sha256(source_path),
    "preprocessing_source_sha256": file_sha256(preprocessing_path),
    "benchmark_preprocessing_source_sha256": file_sha256(
        inputs / "source/datasets/benchmark_preprocessing.py"
    ),
    "detector_report_sha256": file_sha256(detector_report_path),
    "input_package_sha256": quality["input_package_sha256"],
    "attribute_names": config["attribute_names"],
    "environment": {
        "python": sys.version,
        "torch": torch.__version__,
        "torchvision": importlib.metadata.version("torchvision"),
        "Pillow": importlib.metadata.version("Pillow"),
    },
    "device": "cpu",
    "strict_checkpoint_loading": True,
    "all_logits_finite": True,
    "masked_geometry_features_exactly_zero": True,
    "valid_geometry_features_match_mlp": True,
    "invalid_coordinate_invariance_max_abs_error": mask_errors,
    "failure_policy": config["failure_policy"],
    "model_inference_run": True,
    "test_evaluated": False,
    "benchmark_frozen": False,
    "scope": "Four-image validation pipeline smoke; no attribute evaluation.",
    "records": inference_rows,
    "passed": True,
}

destination = output_dir / "m3_pipeline_smoke_report.json"
temporary = destination.with_suffix(".json.tmp")
temporary.write_text(
    json.dumps(report, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)
temporary.replace(destination)

print("Output shape:", tuple(all_logits.shape))
print("Valid geometry:", report["valid_geometry_count"])
print("Masked geometry:", report["masked_geometry_count"])
print("Invalid-coordinate invariance:", mask_errors)
print("Saved:", destination)
print("Day 16 M3 pipeline smoke: PASS")

Training source hash and strict checkpoint loading: PASS
M3 inference: 16/84
M3 inference: 32/84
M3 inference: 48/84
M3 inference: 64/84
M3 inference: 80/84
M3 inference: 84/84
Output shape: (84, 24)
Valid geometry: 82
Masked geometry: 2
Invalid-coordinate invariance: {'large_finite': 0.0, 'nan': 0.0}
Saved: /content/drive/MyDrive/FacialVisualProfile/runs/corruption_preparation/day16/m3_pipeline_smoke_report.json
Day 16 M3 pipeline smoke: PASS


In [8]:
from pathlib import Path
import hashlib
import json
import zipfile
from google.colab import files

folder = Path(
    "/content/drive/MyDrive/FacialVisualProfile/"
    "runs/corruption_preparation/day16"
)
names = [
    "detector_quality_report.json",
    "m3_pipeline_smoke_report.json",
]
payloads = {name: (folder / name).read_bytes() for name in names}
reports = {
    name: json.loads(data) for name, data in payloads.items()
}

for report in reports.values():
    assert report["passed"] is True
    assert report["test_evaluated"] is False

detector_hash = hashlib.sha256(payloads[names[0]]).hexdigest()
assert reports[names[1]]["detector_report_sha256"] == detector_hash

manifest = {
    "day": 16,
    "scope": "Four-image validation pipeline smoke",
    "test_evaluated": False,
    "files": {
        name: hashlib.sha256(data).hexdigest()
        for name, data in payloads.items()
    },
}

destination = Path("/content/day16_reports.zip")
with zipfile.ZipFile(
    destination, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for name, data in payloads.items():
        archive.writestr(name, data)
    archive.writestr(
        "archive_manifest.json",
        json.dumps(manifest, indent=2) + "\n",
    )

print("Day 16 report export: PASS")
files.download(str(destination))

Day 16 report export: PASS


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>